In [3]:
from pathlib import Path

import polars as pl

# ============================================================
# PATHS
# ============================================================

# Notebook is executed from /notebooks
PROJECT_ROOT = Path.cwd().parent.resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "baselines"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "precipitation_amount.parquet"
)

print("=" * 60)
print("BUILD CLIMATOLOGY")
print("=" * 60)
print(f"Project : {PROJECT_ROOT}")
print(f"Input   : {INPUT_FILE}")
print(f"Output  : {OUTPUT_FILE}")
print("=" * 60)

assert INPUT_FILE.exists(), f"Missing input file: {INPUT_FILE}"

BUILD CLIMATOLOGY
Project : C:\Projects\ews\zoryn
Input   : C:\Projects\ews\zoryn\data\processed\observations\master_observations.parquet
Output  : C:\Projects\ews\zoryn\data\processed\baselines\precipitation_amount.parquet


In [4]:
# ============================================================
# LOAD OBSERVATIONS
# ============================================================

observations = pl.scan_parquet(INPUT_FILE)

summary = (
    observations
    .select(
        [
            pl.len().alias("rows"),
            pl.n_unique("h3_id").alias("h3_cells"),
            pl.n_unique("timestamp").alias("timestamps"),
            pl.n_unique("variable").alias("variables"),
            pl.min("timestamp").alias("first_timestamp"),
            pl.max("timestamp").alias("last_timestamp"),
        ]
    )
    .collect()
)

print("=" * 60)
print("INPUT VALIDATION")
print("=" * 60)
print(summary)
print("=" * 60)

assert summary["variables"][0] == 1, "Expected one variable."
assert summary["rows"][0] > 0

INPUT VALIDATION
shape: (1, 6)
┌──────────┬──────────┬────────────┬───────────┬─────────────────┬────────────────┐
│ rows     ┆ h3_cells ┆ timestamps ┆ variables ┆ first_timestamp ┆ last_timestamp │
│ ---      ┆ ---      ┆ ---        ┆ ---       ┆ ---             ┆ ---            │
│ u32      ┆ u32      ┆ u32        ┆ u32       ┆ str             ┆ str            │
╞══════════╪══════════╪════════════╪═══════════╪═════════════════╪════════════════╡
│ 11680044 ┆ 42319    ┆ 276        ┆ 1         ┆ 2002-01         ┆ 2024-12        │
└──────────┴──────────┴────────────┴───────────┴─────────────────┴────────────────┘


In [5]:
# ============================================================
# ADD MONTH
# ============================================================

observations = observations.with_columns(
    pl.col("timestamp")
    .str.slice(5, 2)
    .cast(pl.UInt8)
    .alias("month")
)

month_summary = (
    observations
    .select(
        [
            pl.min("month").alias("min_month"),
            pl.max("month").alias("max_month"),
            pl.n_unique("month").alias("unique_months"),
        ]
    )
    .collect()
)

print("=" * 60)
print("MONTH VALIDATION")
print("=" * 60)
print(month_summary)
print("=" * 60)

assert month_summary["min_month"][0] == 1
assert month_summary["max_month"][0] == 12
assert month_summary["unique_months"][0] == 12

MONTH VALIDATION
shape: (1, 3)
┌───────────┬───────────┬───────────────┐
│ min_month ┆ max_month ┆ unique_months │
│ ---       ┆ ---       ┆ ---           │
│ u8        ┆ u8        ┆ u32           │
╞═══════════╪═══════════╪═══════════════╡
│ 1         ┆ 12        ┆ 12            │
└───────────┴───────────┴───────────────┘


In [6]:
# ============================================================
# BUILD MONTHLY CLIMATOLOGY
# ============================================================

climatology = (
    observations
    .group_by(
        [
            "h3_id",
            "month",
        ]
    )
    .agg(
        [
            pl.col("value").mean().alias("mean"),
            pl.col("value").std().alias("std"),
            pl.col("value").median().alias("median"),
            pl.col("value").quantile(0.05).alias("p05"),
            pl.col("value").quantile(0.10).alias("p10"),
            pl.col("value").quantile(0.25).alias("p25"),
            pl.col("value").quantile(0.75).alias("p75"),
            pl.col("value").quantile(0.90).alias("p90"),
            pl.col("value").quantile(0.95).alias("p95"),
            pl.len().alias("count"),
        ]
    )
    .sort(
        [
            "h3_id",
            "month",
        ]
    )
    .collect()
)

print("=" * 60)
print("CLIMATOLOGY")
print("=" * 60)
print(climatology.head())
print()
print(f"Rows : {len(climatology):,}")
print("=" * 60)

CLIMATOLOGY
shape: (5, 12)
┌─────────────┬───────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬───────┐
│ h3_id       ┆ month ┆ mean       ┆ std        ┆ … ┆ p75        ┆ p90        ┆ p95        ┆ count │
│ ---         ┆ ---   ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---   │
│ str         ┆ u8    ┆ f32        ┆ f32        ┆   ┆ f32        ┆ f32        ┆ f32        ┆ u32   │
╞═════════════╪═══════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════╡
│ 8a7a6325b20 ┆ 1     ┆ 41.211193  ┆ 34.645584  ┆ … ┆ 53.399052  ┆ 81.801918  ┆ 119.630814 ┆ 23    │
│ ffff        ┆       ┆            ┆            ┆   ┆            ┆            ┆            ┆       │
│ 8a7a6325b20 ┆ 2     ┆ 34.273003  ┆ 20.449419  ┆ … ┆ 54.094376  ┆ 68.128143  ┆ 68.492249  ┆ 23    │
│ ffff        ┆       ┆            ┆            ┆   ┆            ┆            ┆            ┆       │
│ 8a7a6325b20 ┆ 3     ┆ 95.590385  ┆ 59.926723  ┆ … ┆ 140.947235

In [7]:
# ============================================================
# QA
# ============================================================

print("=" * 60)
print("CLIMATOLOGY QA")
print("=" * 60)

print(f"Rows          : {len(climatology):,}")
print(f"H3 Cells      : {climatology['h3_id'].n_unique():,}")
print(f"Months        : {climatology['month'].n_unique()}")
print(f"Expected Rows : {42_319 * 12:,}")

assert len(climatology) == 42_319 * 12

assert climatology["month"].min() == 1
assert climatology["month"].max() == 12

assert climatology["count"].min() == 23
assert climatology["count"].max() == 23

print("QA PASSED")
print("=" * 60)

CLIMATOLOGY QA
Rows          : 507,828
H3 Cells      : 42,319
Months        : 12
Expected Rows : 507,828
QA PASSED


In [8]:
# ============================================================
# SAVE
# ============================================================

climatology.write_parquet(
    OUTPUT_FILE,
    compression="zstd",
)

print("=" * 60)
print("CLIMATOLOGY SAVED")
print("=" * 60)
print(f"File : {OUTPUT_FILE}")
print(f"Rows : {len(climatology):,}")
print("=" * 60)

CLIMATOLOGY SAVED
File : C:\Projects\ews\zoryn\data\processed\baselines\precipitation_amount.parquet
Rows : 507,828
